# 002 Feature Assembly (Börde)

**Prerequisite**: run `002_run_feature_pipeline.py` to complete grid generation + data
download + feature extraction.

This notebook is responsible for:
1. Interactive feature selection (edit `FEATURE_CONFIG`)
2. Loading grid + extracted data
3. Assembling `grid_gdf` + `all_node_features_col` per the configuration
4. Saving to `features/assembled/` for downstream graph construction

**Difference from the UK version:** a single region "boerde", with no train/test split.

In [ ]:
import json
import pickle
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

LOCATION_NAME = "boerde"

FEATURES_DIR = Path('./results/intermediate/features')
GRID_DIR = FEATURES_DIR / 'grid'
EXTRACTED_DIR = FEATURES_DIR / 'extracted'
ASSEMBLED_DIR = FEATURES_DIR / 'assembled'
ASSEMBLED_DIR.mkdir(parents=True, exist_ok=True)

print(f'Feature directory: {FEATURES_DIR.resolve()}')

## Feature Selection Configuration

Edit `FEATURE_CONFIG` below to control the feature combination:
- `enabled`: whether to enable this feature group
- `type`: `"numerical"` = numeric columns (merged into grid_gdf), `"array"` = high-dimensional
  arrays (saved separately)
- `columns`: which columns to select

In [ ]:
FEATURE_CONFIG = {
    "landuse": {
        "enabled": True,
        "type": "numerical",
        "source": f"extracted/{LOCATION_NAME}_landuse.npz",
        "columns": [
            "lu_residential_prop", "lu_commercial_prop",
            "lu_industrial_prop", "lu_agricultural_prop", "lu_others_prop"
        ]
    },
    "worldcover": {
        "enabled": True,
        "type": "numerical",
        "source": f"extracted/{LOCATION_NAME}_worldcover.npz",
        "columns": [
            "wc_built_up_ratio", "wc_agricultural_ratio", "wc_others_ratio"
        ]
    },
    "spectral": {
        "enabled": True,
        "type": "array",
        "source": f"extracted/{LOCATION_NAME}_spectral.npy",
        "columns": [
            "NDVI_mean", "NDVI_std", "NDVI_median",
            "NDBI_mean", "NDBI_std", "NDBI_median",
            "NDWI_mean", "NDWI_std", "NDWI_median",
            "BSI_mean", "BSI_std", "BSI_median",
            "UI_mean", "UI_std", "UI_median"
        ]
    },
    "qa": {
        "enabled": True,
        "type": "array",
        "source": f"extracted/{LOCATION_NAME}_qa.npy",
        "columns": [
            "q_residential", "q_commercial", "q_industrial",
            "q_agricultural", "q_others"
        ]
    }
}

# Show the enabled feature groups
for name, cfg in FEATURE_CONFIG.items():
    status = 'ON' if cfg['enabled'] else 'OFF'
    print(f"  [{status}] {name}: type={cfg['type']}, {len(cfg['columns'])} columns")

## Loading and Assembly Functions

In [ ]:
def load_grid(location: str):
    """Load the grid GeoDataFrame and step_size"""
    grid_path = GRID_DIR / f'{location}_grid.gpkg'
    meta_path = GRID_DIR / f'{location}_meta.json'

    grid_gdf = gpd.read_file(str(grid_path))
    with open(meta_path, 'r') as f:
        meta = json.load(f)
    return grid_gdf, meta['step_size_m']


def load_features(config: dict) -> dict:
    """Load feature data per FEATURE_CONFIG"""
    features = {}
    for name, cfg in config.items():
        if not cfg['enabled']:
            continue

        path = FEATURES_DIR / cfg['source']

        if not path.exists():
            print(f"  [Warning] {name}: file not found {path}")
            continue

        if path.suffix == '.npy':
            features[name] = np.load(str(path))
        elif path.suffix == '.npz':
            loaded = np.load(str(path), allow_pickle=True)
            if cfg['type'] == 'numerical':
                if 'agg_data' in loaded:
                    data = loaded['agg_data']
                    col_names = list(loaded['agg_names'])
                else:
                    data = loaded['data']
                    col_names = list(loaded['columns'])
                features[name] = (data, col_names)
            else:
                if 'raw_data' in loaded:
                    features[name] = loaded['raw_data']
                elif 'data' in loaded:
                    features[name] = loaded['data']

    return features


def assemble_grid(grid_gdf: gpd.GeoDataFrame, features: dict, config: dict) -> gpd.GeoDataFrame:
    """Merge all feature columns into grid_gdf"""
    grid_gdf = grid_gdf.copy()

    for name, cfg in config.items():
        if not cfg['enabled'] or name not in features:
            continue

        expected_cols = cfg['columns']

        if cfg['type'] == 'numerical':
            data, col_names = features[name]
            for col in expected_cols:
                if col in col_names:
                    grid_gdf[col] = data[:, col_names.index(col)]
                else:
                    print(f"  [Warning] column '{col}' not found in {name} data")

        elif cfg['type'] == 'array':
            arr = features[name]
            for i, col in enumerate(expected_cols):
                if i < arr.shape[1]:
                    grid_gdf[col] = arr[:, i]
                else:
                    print(f"  [Warning] {name} array only has {arr.shape[1]} columns, skipping '{col}'")

    # Keep only base grid columns + feature columns
    GRID_KEEP_COLS = ['geometry', 'index_region', 'NUTS3', 'Name']
    all_feature_cols = []
    for cfg in config.values():
        if cfg['enabled']:
            all_feature_cols.extend(cfg['columns'])
    keep = GRID_KEEP_COLS + all_feature_cols
    grid_gdf = grid_gdf[[c for c in keep if c in grid_gdf.columns]]

    return grid_gdf

print('Loading functions defined')

## Load + Assemble

In [ ]:
# Load the grid
grid_gdf, step_size_m = load_grid(LOCATION_NAME)
print(f"Grid: {len(grid_gdf)} points, step={step_size_m}m")

# Load features
features = load_features(FEATURE_CONFIG)
for name, fdata in features.items():
    if isinstance(fdata, np.ndarray):
        print(f"  {name}: {fdata.shape}")
    elif isinstance(fdata, tuple):
        print(f"  {name}: {fdata[0].shape}, cols={fdata[1]}")

# Assemble
grid_gdf = assemble_grid(grid_gdf, features, FEATURE_CONFIG)

# Dimension check
n_grid = len(grid_gdf)
for name, fdata in features.items():
    n = fdata.shape[0] if isinstance(fdata, np.ndarray) else fdata[0].shape[0]
    assert n == n_grid, f"{name} dimension mismatch: {n} vs {n_grid}"

print(f"\nAssembly complete: {grid_gdf.shape}")
print(f"Columns: {list(grid_gdf.columns)}")
grid_gdf.head()

## Build the Feature Schema

In [ ]:
# Build the feature schema -- all feature columns are already in grid_gdf
numerical_col_names = []
categorical_col_members = {}

for name, cfg in FEATURE_CONFIG.items():
    if not cfg['enabled']:
        continue
    numerical_col_names.extend(cfg['columns'])

all_node_features_col = [numerical_col_names, categorical_col_members]

print(f'Feature columns ({len(numerical_col_names)}): {numerical_col_names}')
print(f'Categorical features: {categorical_col_members}')

## Save

In [ ]:
# Save
saved_files = []

# grid_points pickle
out_path = ASSEMBLED_DIR / f'{LOCATION_NAME}_grid_points.pickle'
with open(out_path, 'wb') as f:
    pickle.dump([grid_gdf, step_size_m], f)
saved_files.append(out_path.name)

# all_node_features_col
with open(ASSEMBLED_DIR / 'all_node_features_col.pickle', 'wb') as f:
    pickle.dump(all_node_features_col, f)
saved_files.append('all_node_features_col.pickle')

# feature_schema.json
schema = {
    'numerical_col_names': numerical_col_names,
    'categorical_col_members': categorical_col_members,
    'locations': {
        'all': [LOCATION_NAME],
    },
    'region_stats': {
        LOCATION_NAME: {'n_points': len(grid_gdf), 'step_size_m': step_size_m}
    },
}
with open(ASSEMBLED_DIR / 'feature_schema.json', 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)
saved_files.append('feature_schema.json')

print(f'Saved {len(saved_files)} files to {ASSEMBLED_DIR}')
for fn in saved_files:
    print(f'  - {fn}')

## Verification

In [ ]:
# Verification statistics
nan_counts = {col: grid_gdf[col].isna().sum()
              for col in numerical_col_names if col in grid_gdf.columns}
total_nan = sum(nan_counts.values())

print(f'Grid points: {len(grid_gdf):,}')
print(f'Columns: {len(grid_gdf.columns)}')
print(f'Step size: {step_size_m}m')
print(f'Total feature NaNs: {total_nan}')

if total_nan > 0:
    print('\nNaN distribution:')
    for col, cnt in nan_counts.items():
        if cnt > 0:
            print(f'  {col}: {cnt}')

# Numerical statistics
grid_gdf[numerical_col_names].describe()

In [ ]:
# Compatibility check: confirm the pickle format is compatible downstream
test_path = ASSEMBLED_DIR / f'{LOCATION_NAME}_grid_points.pickle'

with open(test_path, 'rb') as f:
    loaded = pickle.load(f)

assert isinstance(loaded, list) and len(loaded) == 2, 'Bad pickle format: expected [grid_gdf, step_size_m]'
assert isinstance(loaded[0], gpd.GeoDataFrame), 'First element should be a GeoDataFrame'
assert isinstance(loaded[1], (int, float)), 'Second element should be numeric (step_size_m)'

print(f'Compatibility check passed: {LOCATION_NAME}')
print(f'  grid_gdf: {loaded[0].shape}, columns={list(loaded[0].columns)}')
print(f'  step_size_m: {loaded[1]}')

# Verify all_node_features_col
with open(ASSEMBLED_DIR / 'all_node_features_col.pickle', 'rb') as f:
    loaded_features = pickle.load(f)
assert isinstance(loaded_features, list) and len(loaded_features) == 2
print(f'\nall_node_features_col check passed')
print(f'  Numerical columns ({len(loaded_features[0])}): {loaded_features[0]}')
print(f'  Categorical columns: {loaded_features[1]}')